In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
#cargar datos
df = pd.read_csv('../data/raw/dataset_fraude_horarios.csv')

In [ ]:

cols_to_drop = ['es_fraude', 'id_transaccion', 'id_usuario', 'fecha']
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
y = df['es_fraude']


categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42, n_jobs=-1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

#entrenar
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

#evaluar
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

/tmp/ipykernel_333034/1246758398.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99     19600
           1       0.87      0.28      0.42       400

    accuracy                           0.98     20000
   macro avg       0.93      0.64      0.71     20000
weighted avg       0.98      0.98      0.98     20000

Confusion Matrix:
 [[19584    16]
 [  289   111]]


In [ ]:


y_pred_proba = model.predict_proba(X_test)[:, 1]

# umbrala  0.2
umbral = 0.2

y_pred_personalizado = (y_pred_proba >= umbral).astype(int)

print("Nuevo Reporte de Clasificación (Umbral 20%):\n", classification_report(y_test, y_pred_personalizado))
print("Nueva Matriz de Confusión:\n", confusion_matrix(y_test, y_pred_personalizado))

Nuevo Reporte de Clasificación (Umbral 20%):
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     19600
           1       0.52      0.75      0.61       400

    accuracy                           0.98     20000
   macro avg       0.76      0.87      0.80     20000
weighted avg       0.99      0.98      0.98     20000

Nueva Matriz de Confusión:
 [[19322   278]
 [  101   299]]


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier



#añadimos smote
smote_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),           # Paso 1: Escalar y codificar variables
    ('smote', SMOTE(random_state=42)),        # Paso 2: Generar fraudes sintéticos
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)) # Paso 3: Entrenar
])

#modelo a entrenar
smote_pipeline.fit(X_train, y_train)

#evaluamos
y_pred_smote = smote_pipeline.predict(X_test)

print(classification_report(y_test, y_pred_smote))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     19600
           1       0.65      0.43      0.52       400

    accuracy                           0.98     20000
   macro avg       0.82      0.71      0.75     20000
weighted avg       0.98      0.98      0.98     20000

